# Notebook 06 — Product Demo & Deployment Layer
**LeaseIQ | AAI-590 Capstone**

This notebook demonstrates the full LeaseIQ product pipeline end-to-end:
1. Load the fine-tuned LegalBERT and XGBoost models
2. Analyze a sample commercial lease contract
3. Walk through the Streamlit UI and FastAPI + Vapi architecture

**Prerequisite:** Run `python scripts/save_xgb_model.py` before this notebook.

## 1. Setup

In [ ]:
import json
import pickle
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "app"))

MODEL_DIR_LB = ROOT / "models" / "legalbert-cuad"
XGB_PATH     = ROOT / "models" / "xgb_risk_model.pkl"
RESULTS_DIR  = ROOT / "results"

print("Project root:", ROOT)
print("LegalBERT path exists:", MODEL_DIR_LB.exists())
print("XGBoost model exists:", XGB_PATH.exists())

## 2. Load Models

In [ ]:
from transformers import pipeline

# LegalBERT as a QA pipeline (CPU for demo)
print("Loading LegalBERT QA pipeline...")
qa_pipeline = pipeline(
    "question-answering",
    model=str(MODEL_DIR_LB),
    tokenizer=str(MODEL_DIR_LB),
    device=-1,
    handle_impossible_answer=True,
)
print("LegalBERT loaded.")

# XGBoost risk classifier
with open(XGB_PATH, "rb") as f:
    xgb_model = pickle.load(f)
print("XGBoost loaded:", type(xgb_model).__name__)

## 3. Sample Contract

We use a short excerpt of a commercial office lease to demonstrate extraction.
In production the Streamlit app accepts full contract PDFs.

In [ ]:
SAMPLE_CONTRACT = """
OFFICE LEASE AGREEMENT

This Office Lease Agreement ("Agreement") is entered into as of January 1, 2024,
by and between Landlord Corp., a Delaware corporation ("Landlord"), and TechStart Inc.,
a California corporation ("Tenant").

1. PREMISES. Landlord hereby leases to Tenant the premises located at 123 Main Street,
   San Francisco, CA 94105, comprising approximately 2,500 square feet on the 4th floor
   (the "Premises").

2. TERM. The initial term of this Lease shall commence on February 1, 2024 and shall
   expire on January 31, 2026 ("Expiration Date"), unless earlier terminated.

3. RENEWAL. Tenant shall have the option to renew this Lease for one (1) additional
   term of two (2) years by providing written notice to Landlord no later than one hundred
   eighty (180) days prior to the Expiration Date.

4. GOVERNING LAW. This Agreement shall be governed by and construed in accordance with
   the laws of the State of California, without regard to its conflict of laws principles.

5. ASSIGNMENT. Tenant shall not assign this Agreement or sublet the Premises or any part
   thereof without the prior written consent of Landlord, which consent shall not be
   unreasonably withheld, conditioned, or delayed.

6. INSURANCE. Tenant shall, at its own cost and expense, procure and maintain throughout
   the term of this Lease commercial general liability insurance with limits of not less
   than One Million Dollars ($1,000,000) per occurrence.
"""

print(f"Sample contract: {len(SAMPLE_CONTRACT):,} characters, ~{len(SAMPLE_CONTRACT.split()):,} words")

## 4. Clause Extraction with LegalBERT

In [ ]:
CLAUSE_CATEGORIES = [
    "Affiliate License-Licensee", "Affiliate License-Licensor", "Agreement Date",
    "Anti-Assignment", "Audit Rights", "Cap On Liability", "Change Of Control",
    "Competitive Restriction Exception", "Covenant Not To Sue", "Document Name",
    "Effective Date", "Exclusivity", "Expiration Date", "Governing Law",
    "Insurance", "Ip Ownership Assignment", "Irrevocable Or Perpetual License",
    "Joint Ip Ownership", "License Grant", "Liquidated Damages",
    "Minimum Commitment", "Most Favored Nation", "No-Solicit Of Customers",
    "No-Solicit Of Employees", "Non-Compete", "Non-Disparagement",
    "Non-Transferable License", "Notice Period To Terminate Renewal", "Parties",
    "Post-Termination Services", "Price Restrictions", "Renewal Term",
    "Revenue/Profit Sharing", "Rofr/Rofo/Rofn", "Source Code Escrow",
    "Termination For Convenience", "Third Party Beneficiary",
    "Uncapped Liability", "Unlimited/All-You-Can-Eat-License",
    "Volume Restriction", "Warranty Duration",
]
CONFIDENCE_THRESHOLD = 0.05

print("Running LegalBERT clause extraction on sample contract...")
clause_results = {}
for cat in CLAUSE_CATEGORIES:
    out     = qa_pipeline(question=cat, context=SAMPLE_CONTRACT)
    answer  = out.get("answer", "") or ""
    score   = float(out.get("score", 0.0))
    present = bool(answer.strip()) and score >= CONFIDENCE_THRESHOLD
    clause_results[cat] = {"answer": answer, "score": score, "present": present}

# Print detected clauses
detected = [(c, v) for c, v in clause_results.items() if v["present"]]
print(f"\nDetected {len(detected)} / {len(CLAUSE_CATEGORIES)} clauses:\n")
for cat, info in sorted(detected, key=lambda x: x[1]["score"], reverse=True):
    print(f"  {cat:<42}  score={info['score']:.3f}  → {info['answer'][:60]!r}")

## 5. Risk Scoring with XGBoost

In [ ]:
HIGH_RISK_CLAUSES = [
    "Cap On Liability", "Governing Law", "Anti-Assignment",
    "Termination For Convenience", "Notice Period To Terminate Renewal",
]
RISK_LABELS = {0: "LOW", 1: "MEDIUM", 2: "HIGH"}

# Build 41-dim binary feature vector
feature_vec = np.array(
    [[1 if clause_results[cat]["present"] else 0 for cat in CLAUSE_CATEGORIES]],
    dtype=np.float32
)

pred  = int(xgb_model.predict(feature_vec)[0])
proba = xgb_model.predict_proba(feature_vec)[0]
tier  = RISK_LABELS[pred]

missing_high = [c for c in HIGH_RISK_CLAUSES if not clause_results[c]["present"]]
present_high = [c for c in HIGH_RISK_CLAUSES if clause_results[c]["present"]]

print(f"Risk Tier: {tier}")
print(f"  P(LOW)    = {proba[0]:.1%}")
print(f"  P(MEDIUM) = {proba[1]:.1%}")
print(f"  P(HIGH)   = {proba[2]:.1%}")
print()
print("Critical clauses PRESENT:", present_high if present_high else "none")
print("Critical clauses MISSING:", missing_high if missing_high else "none")

## 6. SHAP Explainability

In [ ]:
import shap

explainer = shap.TreeExplainer(xgb_model)
sv = np.array(explainer.shap_values(feature_vec))

# Handle 3D SHAP output from XGBoost 3.x
if sv.ndim == 3 and sv.shape[0] == 3:
    shap_high = sv[2][0]
elif sv.ndim == 3 and sv.shape[-1] == 3:
    shap_high = sv[0, :, 2]
else:
    shap_high = sv[0]

shap_dict = dict(zip(CLAUSE_CATEGORIES, shap_high.tolist()))
shap_sorted = sorted(shap_dict.items(), key=lambda kv: abs(kv[1]), reverse=True)[:15]

print("Top 15 risk drivers (SHAP values for HIGH class):")
for cat, val in shap_sorted:
    direction = "↑ HIGH" if val > 0 else "↓ LOW"
    present_str = "✓" if clause_results[cat]["present"] else "✗"
    print(f"  {present_str} {cat:<42}  SHAP = {val:+.4f}  {direction}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
cats   = [s[0] for s in shap_sorted[::-1]]
vals   = [s[1] for s in shap_sorted[::-1]]
colors = ["#C44E52" if v > 0 else "#4C72B0" for v in vals]
ax.barh(cats, vals, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("SHAP value (impact on HIGH risk)")
ax.set_title("Top 15 Risk Drivers — Sample Contract")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "product_demo_shap.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved → results/product_demo_shap.png")

## 7. Full Result Summary

In [ ]:
result = {
    "risk": {
        "risk_label": tier,
        "prob_low":    round(proba[0]*100, 1),
        "prob_medium": round(proba[1]*100, 1),
        "prob_high":   round(proba[2]*100, 1),
    },
    "missing_high_risk": missing_high,
    "present_high_risk": present_high,
    "top_shap":          {k: round(v, 4) for k, v in shap_sorted[:5]},
    "clauses_detected":  len(detected),
    "clauses_total":     len(CLAUSE_CATEGORIES),
}

print(json.dumps(result, indent=2))

## 8. Product Architecture

```
                        ┌──────────────────────┐
                        │   User (web / voice) │
                        └────────┬─────────────┘
                                 │
                    ┌────────────┴──────────────┐
                    │                           │
            ┌───────▼───────┐        ┌─────────▼──────────┐
            │  Streamlit UI │        │  Vapi Voice Agent  │
            │  (port 8501)  │        │  (telephony / web) │
            └───────┬───────┘        └─────────┬──────────┘
                    │                          │  POST /vapi/webhook
                    │                          │
                    └────────┬─────────────────┘
                             │
                    ┌────────▼────────┐
                    │ FastAPI Backend │
                    │   (port 8000)   │
                    └────────┬────────┘
                             │
                    ┌────────▼────────────────────┐
                    │   app/inference.py          │
                    │   ┌─────────────────────┐   │
                    │   │  LegalBERT QA pipe  │   │  ← clause extraction
                    │   │  (41 clause types)  │   │
                    │   └─────────────────────┘   │
                    │   ┌─────────────────────┐   │
                    │   │  XGBoost classifier │   │  ← risk scoring
                    │   │  (LOW/MED/HIGH)     │   │
                    │   └─────────────────────┘   │
                    │   ┌─────────────────────┐   │
                    │   │  SHAP explainer     │   │  ← feature importance
                    │   └─────────────────────┘   │
                    └─────────────────────────────┘
```

### Running the Streamlit UI
```bash
source leaseiq-env/bin/activate
streamlit run app/streamlit_app.py
# Open http://localhost:8501
```

### Running the FastAPI + Vapi backend
```bash
source leaseiq-env/bin/activate
uvicorn app.api:app --reload --port 8000
# For Vapi: expose with ngrok tunnel:
#   ngrok http 8000
# Then update vapi_config.json serverUrl with the ngrok URL.
```